In [1]:
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
import json
from collections import Counter

BASE_DIR  = Path("../data/processed")
SYNTH_DIR = Path("../data/synthetic/GAN")
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FEATURE_COLS = [
    "op1","op2",
    "s2","s3","s4","s7","s8","s9",
    "s11","s12","s13","s14","s15","s17","s20","s21",
]

SEQ_LEN     = 30
INPUT_DIM   = 16
HIDDEN_DIM  = 128
LATENT_DIM  = 64
NUM_CLASSES = 4
EMBED_DIM   = 16
SUBSETS     = ["FD001","FD002","FD003","FD004"]

print(f"Device : {DEVICE}")


Device : cpu


In [2]:
class CGANGenerator(nn.Module):
    def __init__(self, latent_dim, hidden_dim, seq_len,
                 output_dim, num_classes, embed_dim):
        super().__init__()
        self.seq_len     = seq_len
        self.class_embed = nn.Embedding(num_classes, embed_dim)
        self.fc_in  = nn.Sequential(
            nn.Linear(latent_dim + embed_dim, hidden_dim),
            nn.LeakyReLU(0.2),
        )
        self.gru    = nn.GRU(hidden_dim, hidden_dim, num_layers=2, batch_first=True)
        self.fc_out = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim // 2, output_dim),
            nn.Sigmoid(),
        )
    def forward(self, z, c):
        emb   = self.class_embed(c)
        inp   = torch.cat([z, emb], dim=-1)
        h     = self.fc_in(inp)
        h_seq = h.unsqueeze(1).repeat(1, self.seq_len, 1)
        h_seq = h_seq + 0.1 * torch.randn_like(h_seq)
        out, _ = self.gru(h_seq)
        return self.fc_out(out)


def load_generator(ckpt_dir):
    G = CGANGenerator(LATENT_DIM, HIDDEN_DIM, SEQ_LEN,
                      INPUT_DIM, NUM_CLASSES, EMBED_DIM).to(DEVICE)
    G.load_state_dict(torch.load(ckpt_dir / "generator.pt", map_location=DEVICE))
    G.eval()
    return G


def generate_samples(G, n_samples, class_id):
    all_samples = []
    batch_size  = 128
    generated   = 0
    with torch.no_grad():
        while generated < n_samples:
            bs    = min(batch_size, n_samples - generated)
            z     = torch.randn(bs, LATENT_DIM).to(DEVICE)
            c     = torch.full((bs,), class_id, dtype=torch.long).to(DEVICE)
            X_hat = G(z, c).cpu().numpy()
            all_samples.append(X_hat)
            generated += bs
    return np.concatenate(all_samples, axis=0)


print("Generator and helpers defined.")


Generator and helpers defined.


In [3]:
all_X_balanced      = []
all_labels_balanced = []

for subset in SUBSETS:
    print(f"\nProcessing {subset}...")

    data_dir = BASE_DIR / subset
    X_real   = np.load(data_dir / "X_train.npy")
    labels   = np.load(data_dir / "labels_train.npy")

    real_counts  = Counter(labels.tolist())
    target_count = max(real_counts.values())

    print(f"  Real counts : {dict(sorted(real_counts.items()))}")
    print(f"  Target      : {target_count}")

    # clean path resolution
    if subset == "FD001":
        ckpt_dir = BASE_DIR / "FD001" / "checkpoints_cgan"
    else:
        ckpt_dir = BASE_DIR / subset / f"checkpoints_cgan_{subset}"

    print(f"  Loading from: {ckpt_dir}")
    G = load_generator(ckpt_dir)

    synth_X      = []
    synth_labels = []

    for c in range(NUM_CLASSES):
        needed = max(0, target_count - real_counts.get(c, 0))
        if needed == 0:
            print(f"    C{c}: skipping (majority class)")
            continue
        print(f"    C{c}: generating {needed} samples...")
        samples = generate_samples(G, needed, c)
        synth_X.append(samples)
        synth_labels.append(np.full(needed, c, dtype=np.int64))

    if synth_X:
        synth_X_arr  = np.concatenate(synth_X,      axis=0)
        synth_lb_arr = np.concatenate(synth_labels, axis=0)
    else:
        synth_X_arr  = np.empty((0, SEQ_LEN, INPUT_DIM), dtype=np.float32)
        synth_lb_arr = np.empty((0,),                     dtype=np.int64)

    subset_synth = SYNTH_DIR / subset
    subset_synth.mkdir(parents=True, exist_ok=True)
    np.save(subset_synth / "synth_X.npy",      synth_X_arr)
    np.save(subset_synth / "synth_labels.npy", synth_lb_arr)

    X_bal = np.concatenate([X_real, synth_X_arr], axis=0)
    y_bal = np.concatenate([labels, synth_lb_arr], axis=0)
    idx   = np.random.permutation(len(X_bal))
    X_bal = X_bal[idx]
    y_bal = y_bal[idx]

    np.save(data_dir / "X_balanced.npy",      X_bal)
    np.save(data_dir / "labels_balanced.npy", y_bal)

    all_X_balanced.append(X_bal)
    all_labels_balanced.append(y_bal)

    print(f"  Balanced: {X_bal.shape}  dist={Counter(y_bal.tolist())}")

print("\nAll subsets generated and balanced.")


Processing FD001...
  Real counts : {0: 7633, 1: 4998, 2: 4000, 3: 1100}
  Target      : 7633
  Loading from: ..\data\processed\FD001\checkpoints_cgan
    C0: skipping (majority class)
    C1: generating 2635 samples...
    C2: generating 3633 samples...
    C3: generating 6533 samples...
  Balanced: (30532, 30, 16)  dist=Counter({1: 7633, 0: 7633, 3: 7633, 2: 7633})

Processing FD002...
  Real counts : {0: 19962, 1: 12997, 2: 10400, 3: 2860}
  Target      : 19962
  Loading from: ..\data\processed\FD002\checkpoints_cgan_FD002
    C0: skipping (majority class)
    C1: generating 6965 samples...
    C2: generating 9562 samples...
    C3: generating 17102 samples...
  Balanced: (79848, 30, 16)  dist=Counter({1: 19962, 2: 19962, 0: 19962, 3: 19962})

Processing FD003...
  Real counts : {0: 11720, 1: 5000, 2: 4000, 3: 1100}
  Target      : 11720
  Loading from: ..\data\processed\FD003\checkpoints_cgan_FD003
    C0: skipping (majority class)
    C1: generating 6720 samples...
    C2: genera

In [4]:
import sys; sys.path.insert(0, '..')
from src.utils.dataset_balancing import build_unified_dataset

# merge all 4 balanced datasets into one unified training set capping subset size (BUG-010 fix)
subset_data = list(zip(all_X_balanced, all_labels_balanced))
X_unified, y_unified = build_unified_dataset(subset_data)

UNIFIED_DIR = Path("../data/processed/unified")
UNIFIED_DIR.mkdir(parents=True, exist_ok=True)

np.save(UNIFIED_DIR / "X_train_unified.npy",      X_unified)
np.save(UNIFIED_DIR / "labels_train_unified.npy", y_unified)

print(f"Unified dataset shape : {X_unified.shape}")
print(f"Unified class dist    : {Counter(y_unified.tolist())}")
print(f"Saved to              : {UNIFIED_DIR}")


Unified dataset shape : (272784, 30, 16)
Unified class dist    : Counter({3: 68196, 1: 68196, 2: 68196, 0: 68196})
Saved to              : ..\data\processed\unified
